# Test

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-05-01 13:32:04,205] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-05-01 13:32:04,206] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

Initialization complete


## Additional

In [8]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx
import logging

# Open Model

In [9]:
s3s.OpenModel(dbName=r"../Tutorial051_Assets/Toolkit_Tutorial51_Model.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

Model is open for further operation


In [10]:
object_types = [item for item in dir(s3s.ObjectTypes) if not (item.startswith('__') and item.endswith('__'))]
print(object_types)

['AGSN_HydraulicProfile', 'AirVessel', 'Arrow', 'Atmosphere', 'BlockConnectionNode', 'CalcPari', 'CharacteristicLossTable', 'CharacteristicLossTable_Row', 'Circle', 'Compressor', 'CompressorTable', 'CompressorTable_Row', 'ControlEngineeringNexus', 'ControlMode', 'ControlPointTable', 'ControlPointTable_Row', 'ControlValve', 'ControlVariableConverter', 'ControlVariableConverterRSTE', 'CrossSectionTable', 'CrossSectionTable_Row', 'DPGR_DPKT_DatapointDpgrConnection', 'DPGR_DataPointGroup', 'DPKT_Datapoint', 'DamageRatesTable', 'DamageRatesTable_Row', 'DeadTimeElement', 'Demand', 'DifferentialRegulator', 'DirectionalArrow', 'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Divider', 'DriveEfficiencyTable', 'DriveEfficiencyTable_Row', 'DrivePowerTable', 'DrivePowerTable_Row', 'EBES_FeederGroups', 'EfficiencyConverterTable', 'EfficiencyConverterTable_Row', 'ElementQuery', 'EnergyRecoveryTable', 'EnergyRecoveryTable_Row', 'EnvironmentTemp', 'FWBZ_DistrictHeatingReferenceValues', 'FlapValve'

In [15]:
node1 = s3s.AddNewNode(tkCont=s3s.GetMainContainer()[0], name="1", x=0, y=0, z=0, qm_PH=0.7071, symbolFactor=1.0, description="", idRef="-1", kvr=1, typ="PKON")

New node added


In [ ]:
node2 = s3s.AddNewNode(tkCont=s3s.GetMainContainer()[0], name="1", x=0, y=0, z=0, qm_PH=0.7071, symbolFactor=1.0, description="", idRef="-1", kvr=1, typ="PKON")

In [12]:
connection_elements = []

In [ ]:
import contextlib
import io
import logging

# Build a viability mapping for object types that can be connected between two nodes.
connectable_type_map = {}
viable_object_types = []
non_viable_details = {}

# Messages that indicate the connect call failed although no Python exception is raised.
failure_markers = [
    "nicht an Knoten KI angeschlossen",
    "nicht an Knoten K angeschlossen",
    "nicht angeschlossen",
    "could not",
    "cannot",
    "error",
]

# Reduce noisy output from failed probe attempts.
logging.disable(logging.CRITICAL)

for ot in object_types:
    with contextlib.redirect_stdout(io.StringIO()) as out_buf, contextlib.redirect_stderr(io.StringIO()) as err_buf:
        try:
            node_i = s3s.AddNewNode(
                tkCont=s3s.GetMainContainer()[0],
                name=f"probe_i_{ot}",
                x=0,
                y=0,
                z=0,
                qm_PH=0.7071,
                symbolFactor=1.0,
                description="",
                idRef="-1",
                kvr=1,
                typ="PKON",
            )
            node_k = s3s.AddNewNode(
                tkCont=s3s.GetMainContainer()[0],
                name=f"probe_k_{ot}",
                x=0,
                y=0,
                z=0,
                qm_PH=0.7071,
                symbolFactor=1.0,
                description="",
                idRef="-1",
                kvr=1,
                typ="PKON",
            )

            elem = s3s.InsertElement(s3s.ObjectTypes[ot], IdRef="-1")
            s3s.ConnectConnectingElementWithNodes(Tk=elem, keyOfNodeI=node_i, keyOfNodeK=node_k)
        except Exception as ex:
            connectable_type_map[ot] = False
            non_viable_details[ot] = f"Exception: {type(ex).__name__}: {ex}"
            continue

    probe_output = (out_buf.getvalue() + "\n" + err_buf.getvalue()).strip()
    lower_output = probe_output.lower()
    has_failure_text = any(marker in lower_output for marker in failure_markers)

    if has_failure_text:
        connectable_type_map[ot] = False
        non_viable_details[ot] = probe_output
    else:
        connectable_type_map[ot] = True
        viable_object_types.append(ot)

logging.disable(logging.NOTSET)

print(f"Viable connectable object types: {len(viable_object_types)} / {len(object_types)}")
for ot in viable_object_types:
    print(ot)

print("\nNon-viable examples (first 10):")
for ot in sorted(non_viable_details)[:10]:
    first_line = non_viable_details[ot].splitlines()[0] if non_viable_details[ot] else "<no message>"
    print(f"- {ot}: {first_line}")

Viable connectable object types: 149 / 149
AGSN_HydraulicProfile
AirVessel
Arrow
Atmosphere
BlockConnectionNode
CalcPari
CharacteristicLossTable
CharacteristicLossTable_Row
Circle
Compressor
CompressorTable
CompressorTable_Row
ControlEngineeringNexus
ControlMode
ControlPointTable
ControlPointTable_Row
ControlValve
ControlVariableConverter
ControlVariableConverterRSTE
CrossSectionTable
CrossSectionTable_Row
DPGR_DPKT_DatapointDpgrConnection
DPGR_DataPointGroup
DPKT_Datapoint
DamageRatesTable
DamageRatesTable_Row
DeadTimeElement
Demand
DifferentialRegulator
DirectionalArrow
DistrictHeatingConsumer
DistrictHeatingFeeder
Divider
DriveEfficiencyTable
DriveEfficiencyTable_Row
DrivePowerTable
DrivePowerTable_Row
EBES_FeederGroups
EfficiencyConverterTable
EfficiencyConverterTable_Row
ElementQuery
EnergyRecoveryTable
EnergyRecoveryTable_Row
EnvironmentTemp
FWBZ_DistrictHeatingReferenceValues
FlapValve
FlowControlUnit
FluidQualityParamSet
FluidQualityParamSet_OS
FluidThermalPropertyGroup
FreeDuc

In [20]:
print(viable_object_types)

['AGSN_HydraulicProfile', 'AirVessel', 'Arrow', 'Atmosphere', 'BlockConnectionNode', 'CalcPari', 'CharacteristicLossTable', 'CharacteristicLossTable_Row', 'Circle', 'Compressor', 'CompressorTable', 'CompressorTable_Row', 'ControlEngineeringNexus', 'ControlMode', 'ControlPointTable', 'ControlPointTable_Row', 'ControlValve', 'ControlVariableConverter', 'ControlVariableConverterRSTE', 'CrossSectionTable', 'CrossSectionTable_Row', 'DPGR_DPKT_DatapointDpgrConnection', 'DPGR_DataPointGroup', 'DPKT_Datapoint', 'DamageRatesTable', 'DamageRatesTable_Row', 'DeadTimeElement', 'Demand', 'DifferentialRegulator', 'DirectionalArrow', 'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Divider', 'DriveEfficiencyTable', 'DriveEfficiencyTable_Row', 'DrivePowerTable', 'DrivePowerTable_Row', 'EBES_FeederGroups', 'EfficiencyConverterTable', 'EfficiencyConverterTable_Row', 'ElementQuery', 'EnergyRecoveryTable', 'EnergyRecoveryTable_Row', 'EnvironmentTemp', 'FWBZ_DistrictHeatingReferenceValues', 'FlapValve'

In [11]:
1/0

ZeroDivisionError: division by zero